In [1]:
import re
import math
import random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

In [3]:
with open("corpus.txt", "r") as file:
    sents = to_sentences(file.read())

rng = random.Random(0)
rng.shuffle(sents)

cut = int(0.8 * len(sents))

train = sents[:cut]
test = sents[cut:]

print("Total sentences:", len(sents))
print("Training sentences:", len(train))
print("Testing sentences:", len(test))

Total sentences: 60
Training sentences: 48
Testing sentences: 12


In [4]:
def build(sents, n):
    """Map each (n-1)-word context to a Counter of following words."""
    foll = defaultdict(Counter)

    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]

        for i in range(len(toks) - n + 1):
            context = tuple(toks[i:i + n - 1])
            next_word = toks[i + n - 1]
            foll[context][next_word] += 1

    return foll

In [5]:
V = set(
    token
    for s in train
    for token in to_tokens(s)
) | {"</s>", "<unk>"}

print("Training vocabulary size:", len(V))

Training vocabulary size: 145


In [6]:
bi = build(train, 2)
tri = build(train, 3)

print("Bigram contexts:", len(bi))
print("Trigram contexts:", len(tri))

Bigram contexts: 144
Trigram contexts: 288


In [7]:
def perplexity(foll, n, data, k=1.0):
    log_sum = 0.0
    N = 0

    for s in data:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]

        for i in range(len(toks) - n + 1):
            ctx = tuple(toks[i:i + n - 1])
            w = toks[i + n - 1]

            c = foll.get(ctx, Counter())
            total = sum(c.values())

            p = (c[w] + k) / (total + k * len(V))

            log_sum += math.log(p)
            N += 1

    return math.exp(-log_sum / N)

In [8]:
bi_train = perplexity(bi, 2, train)
bi_test = perplexity(bi, 2, test)

tri_train = perplexity(tri, 3, train)
tri_test = perplexity(tri, 3, test)

print(
    f"bigram  perplexity  ->  "
    f"train {bi_train:6.1f}   held-out {bi_test:6.1f}"
)

print(
    f"trigram perplexity  ->  "
    f"train {tri_train:6.1f}   held-out {tri_test:6.1f}"
)

bigram  perplexity  ->  train   52.7   held-out   94.1
trigram perplexity  ->  train   60.6   held-out  115.8


In [9]:
print("BIGRAM")
print("Training cross-entropy:", round(math.log(bi_train), 4))
print("Held-out cross-entropy:", round(math.log(bi_test), 4))

print("\nTRIGRAM")
print("Training cross-entropy:", round(math.log(tri_train), 4))
print("Held-out cross-entropy:", round(math.log(tri_test), 4))

BIGRAM
Training cross-entropy: 3.9645
Held-out cross-entropy: 4.5448

TRIGRAM
Training cross-entropy: 4.1045
Held-out cross-entropy: 4.7522


In [10]:
for k in [0.01, 0.1, 0.5, 1.0, 2.0]:
    print(
        f"k={k:<4} | "
        f"Bigram: {perplexity(bi, 2, test, k):.2f} | "
        f"Trigram: {perplexity(tri, 3, test, k):.2f}"
    )

k=0.01 | Bigram: 88.49 | Trigram: 118.43
k=0.1  | Bigram: 69.96 | Trigram: 102.02
k=0.5  | Bigram: 83.63 | Trigram: 109.56
k=1.0  | Bigram: 94.14 | Trigram: 115.84
k=2.0  | Bigram: 105.83 | Trigram: 122.74


## Observations — Lesson 6

1. Cross-entropy measures the average negative log probability
   assigned to the actual next words in a text sequence.

2. Perplexity is the exponential of cross-entropy and measures
   how surprised the model is by the text.

3. Lower perplexity indicates better predictive performance
   on the evaluated dataset.

4. Training perplexity is lower than held-out perplexity,
   showing that the model performs better on familiar text.

5. The trigram has worse held-out perplexity than the bigram
   on this small corpus because its longer contexts are sparse.

6. More context does not automatically improve prediction.
   A model needs sufficient training data to learn reliable
   probabilities for longer contexts.

7. Evaluation must use held-out data rather than only the
   training corpus to measure generalization.